# Dynamic Programming — Subtopic 7

## DP on LIS (Longest Increasing Subsequence) family

**Problems covered:**
1. LIS (standard $O(n^2)$)
2. Print LIS (reconstruction)
3. Number of LIS
4. Longest Bitonic Subsequence
5. LIS via patience sorting ($O(n \log n)$)
6. Longest Divisible Subset
7. Russian Doll Envelopes

> The defining state for this family is "**LIS ending exactly at index $i$**" — the same "ending here" anchor we saw for Longest Common Substring (§5.3). This invariant is what lets a single index $i$ serve as state, even though the answer is a subsequence problem.
>
> The centerpiece is the **patience sorting $O(n \log n)$ algorithm**, which we derive from first principles by asking "what is the optimal predecessor to remember at each length?"

---

# The state — "LIS ending exactly at $i$"

## Why a prefix-only state fails

A first instinct: $\text{dp}[i]$ = LIS in the prefix $\text{nums}[0..i]$. This **fails to compose**. To extend a subsequence at position $i+1$ we need to know its *last element*, not just its length. A prefix-only state hides that information.

## The fix: anchor on "ending at $i$"

$$\text{dp}[i] = \text{length of the longest increasing subsequence that ends exactly at index}\ i\ (\text{i.e., uses}\ \text{nums}[i]\ \text{as its last element})$$

This is a familiar pattern — exactly the trick from Longest Common Substring. By anchoring the subsequence to end at $i$, the recurrence becomes Markovian in $i$: to compute $\text{dp}[i]$, we look at all *earlier indices $j < i$* with $\text{nums}[j] < \text{nums}[i]$ and pick the best.

> **Important:** the answer to "what is the LIS of the whole array" is **not** $\text{dp}[n-1]$. It's $\max_i \text{dp}[i]$, since the LIS may end anywhere. Same gotcha as Longest Common Substring's "scan the whole table."

## The two complexity regimes

| Regime | Algorithm | When |
|---|---|---|
| $O(n^2)$ | DP with explicit inner loop over $j < i$ | small $n$, easy reconstruction, or generalizations (divisibility, count) |
| $O(n \log n)$ | Patience sorting via binary search on `tails[]` | large $n$, length-only queries (reconstruction is harder) |

The first regime is the universal hammer — it generalizes to almost any LIS variant (count, bitonic, divisible, etc.). The second is the speed-of-light algorithm for the *vanilla length* question, with a beautiful invariant we'll prove.

---

## 7.1 Longest Increasing Subsequence (standard $O(n^2)$)

**Problem.** Given $\text{nums}[0..n-1]$, find the length of the longest strictly increasing subsequence.

---

### Theory

**State definition.**
$\text{dp}[i]$ = length of LIS ending exactly at index $i$ (using $\text{nums}[i]$ as the last element).

**Invariant.** Every increasing subsequence ending at $i$ either consists of $\text{nums}[i]$ alone (length 1) or extends some increasing subsequence ending at an earlier index $j < i$ where $\text{nums}[j] < \text{nums}[i]$. The recurrence picks the best such predecessor.

**Recurrence.**
$$\text{dp}[i] = 1 + \max\!\Big(\ \{0\}\ \cup\ \{\text{dp}[j]\ :\ 0 \le j < i,\ \text{nums}[j] < \text{nums}[i]\}\ \Big)$$

(The $\{0\}$ in the max ensures $\text{dp}[i] \ge 1$ for the trivial single-element subsequence.)

**Base case.** $\text{dp}[0] = 1$.

**Answer.** $\max_i \text{dp}[i]$.

**Boundary transitions table.**

| Case | Recurrence value |
|---|---|
| $i = 0$ | $1$ (the trivial single-element subsequence) |
| $\nexists j < i$ with $\text{nums}[j] < \text{nums}[i]$ | $1$ |
| $\exists j < i$ with $\text{nums}[j] < \text{nums}[i]$ | $1 + \max\{\text{dp}[j]\}$ over valid $j$ |

**Why it works.**
- *Optimal substructure:* An optimal LIS ending at $i$, dropping its last element, is an optimal LIS ending at some predecessor $j$. (Otherwise we could swap in a longer prefix LIS, contradicting optimality.) So $\text{dp}[i]$ reduces to a known smaller subproblem.
- *Overlapping subproblems:* naively, the number of increasing subsequences is exponential. The DP has $n$ states.

**Complexity.** Time $O(n^2)$ (each $i$ scans $j = 0..i-1$). Space $O(n)$.

**Why no space optimization to $O(1)$.** Unlike 1D DPs where $\text{dp}[i]$ depends on a constant-width window (e.g., $\text{dp}[i-1], \text{dp}[i-2]$ in Climbing Stairs), here $\text{dp}[i]$ depends on *all* prior values $\text{dp}[0..i-1]$. The dependency window is the entire history. No constant-space reduction is possible at the DP level. (The $O(n \log n)$ patience sorting algorithm — covered in §7.5 — is a different algorithm with the same $O(n)$ space.)

**Delta.** This is the kernel. The next four problems all build on this $\text{dp}[i]$ in essentially the same way, with one twist each.

---

In [ ]:
// LIS standard — three implementations
#include <vector>
#include <algorithm>
#include <iostream>
using namespace std;

// (A) Top-down memoization with state (i, prev_index).
//     f(i, prev) = LIS in nums[i..n-1] given the previous index taken is `prev` (or -1 for "none yet").
//     This is a different state shape than the bottom-up dp[i] = "ending at i" — both are valid.
//     The (i, prev) state has size O(n^2), so memoization makes time O(n^2) — same as tabulation.
int lis_memo_helper(int i, int prev, const vector<int>& nums, vector<vector<int>>& dp) {
    int n = (int)nums.size();
    if (i == n) return 0;                                  // no more elements
    // Map prev = -1 to dp index n (sentinel column for "no previous")
    int prevIdx = (prev == -1) ? n : prev;
    if (dp[i][prevIdx] != -1) return dp[i][prevIdx];
    int notTake = lis_memo_helper(i + 1, prev, nums, dp);
    int take = 0;
    if (prev == -1 || nums[prev] < nums[i])                // strict-increasing constraint
        take = 1 + lis_memo_helper(i + 1, i, nums, dp);
    dp[i][prevIdx] = max(notTake, take);
    return dp[i][prevIdx];
}
int lis_memo(const vector<int>& nums) {
    int n = (int)nums.size();
    if (n == 0) return 0;
    vector<vector<int>> dp(n, vector<int>(n + 1, -1));     // n+1 columns: prev ∈ {0,..,n-1} ∪ {sentinel}
    return lis_memo_helper(0, -1, nums, dp);
}

// (B) Tabulation — the "ending here" formulation.
//     dp[i] = LIS length ending exactly at index i.
int lis_tab(const vector<int>& nums) {
    int n = (int)nums.size();
    if (n == 0) return 0;
    vector<int> dp(n, 1);                                  // base: every i has at least the singleton subsequence
    int best = 1;
    for (int i = 1; i < n; ++i) {
        for (int j = 0; j < i; ++j) {                      // scan all earlier indices
            if (nums[j] < nums[i] && dp[j] + 1 > dp[i]) {  // strictly less + improves length
                dp[i] = dp[j] + 1;                         // recurrence: extend predecessor's LIS
            }
        }
        if (dp[i] > best) best = dp[i];                    // running max over endpoints
    }
    return best;
}

// (C) "Optimized" version: same O(n^2) time and O(n) space; we use it for parity with other notebooks.
//     There is no true O(1)-space optimization at this DP level; the inner max needs the full history.
//     For the genuine O(n log n) speedup, see §7.5 (patience sorting).
int lis_opt(const vector<int>& nums) {
    return lis_tab(nums);                                  // same algorithm; placeholder for symmetry
}


In [ ]:
// Tests — LIS standard
auto run_lis = [](vector<int> nums, int expected) {
    int a = lis_memo(nums);
    int b = lis_tab(nums);
    int c = lis_opt(nums);
    bool ok = (a == expected && b == expected && c == expected);
    cout << "nums=[";
    for (size_t k = 0; k < nums.size(); ++k) cout << nums[k] << (k+1 < nums.size() ? "," : "");
    cout << "] -> memo=" << a << " tab=" << b << " opt=" << c
         << " (Expected: " << expected << ")"
         << (ok ? " OK" : " FAIL") << '\n';
};

run_lis({},                          0);    // empty
run_lis({5},                         1);    // single
run_lis({1, 2, 3, 4, 5},             5);    // strictly increasing: whole array
run_lis({5, 4, 3, 2, 1},             1);    // strictly decreasing: just any single
run_lis({2, 2, 2, 2},                1);    // all same: strict requires only one
run_lis({10, 9, 2, 5, 3, 7, 101, 18}, 4);  // classic LeetCode: 2,5,7,101 or 2,3,7,101
run_lis({0, 1, 0, 3, 2, 3},          4);    // 0,1,2,3
run_lis({7, 7, 7, 7, 7, 7, 7},       1);
run_lis({1, 3, 6, 7, 9, 4, 10, 5, 6}, 6);  // 1,3,6,7,9,10


## 7.2 Print LIS

**Problem.** Same input; return *one* longest increasing subsequence (any valid one if multiple exist).

---

### Theory — parent pointers

Alongside $\text{dp}[i]$, maintain $\text{parent}[i]$ = the predecessor index $j$ that achieved $\text{dp}[i] = \text{dp}[j] + 1$, or $-1$ if $\text{dp}[i] = 1$ (no predecessor).

**Reconstruction algorithm.**
1. Run the standard $O(n^2)$ LIS DP, populating both $\text{dp}[]$ and $\text{parent}[]$.
2. Find the index $i^* = \arg\max_i \text{dp}[i]$ (any winner if ties).
3. Walk backwards from $i^*$ via parent pointers: emit $\text{nums}[i^*]$, then $\text{nums}[\text{parent}[i^*]]$, then $\text{nums}[\text{parent}[\text{parent}[i^*]]]$, etc., until we hit $-1$.
4. Reverse the emitted list.

**Correctness.** At step $i$ during the DP, $\text{parent}[i]$ stores a $j$ that witnessed $\text{dp}[i] = \text{dp}[j] + 1$. By induction on length, walking parents from $i$ produces an actual LIS of length $\text{dp}[i]$ ending at $i$. Picking $i^* = \arg\max$ retrieves a global LIS.

**Complexity.** Time $O(n^2)$ for the DP, $O(n)$ for the walk. Space $O(n)$ for both arrays.

**Delta vs LIS.** Just adds the parent array. The DP loop is unchanged in structure — same scan over $j < i$ — only it records *which* $j$ won.

---

In [ ]:
// Print LIS — reconstruction via parent pointers
#include <vector>
#include <algorithm>
#include <iostream>
using namespace std;

vector<int> printLIS(const vector<int>& nums) {
    int n = (int)nums.size();
    if (n == 0) return {};
    vector<int> dp(n, 1);
    vector<int> parent(n, -1);                             // parent[i] = predecessor that won, -1 if none
    int bestIdx = 0;
    for (int i = 1; i < n; ++i) {
        for (int j = 0; j < i; ++j) {
            if (nums[j] < nums[i] && dp[j] + 1 > dp[i]) {
                dp[i] = dp[j] + 1;
                parent[i] = j;                             // record the winning predecessor
            }
        }
        if (dp[i] > dp[bestIdx]) bestIdx = i;              // track global-best endpoint
    }
    // Walk back via parents, collecting indices.
    vector<int> result;
    for (int i = bestIdx; i != -1; i = parent[i]) {
        result.push_back(nums[i]);
    }
    reverse(result.begin(), result.end());                 // built back-to-front
    return result;
}


In [ ]:
// Tests — Print LIS
auto run_pLIS = [](vector<int> nums, int expectedLen) {
    vector<int> r = printLIS(nums);
    // Validate: r is strictly increasing AND r is a subsequence of nums AND |r| == expectedLen
    bool incr = true;
    for (size_t k = 1; k < r.size(); ++k) if (r[k] <= r[k-1]) incr = false;
    auto isSubseq = [&](const vector<int>& sub, const vector<int>& seq) {
        size_t k = 0;
        for (int x : seq) if (k < sub.size() && x == sub[k]) ++k;
        return k == sub.size();
    };
    bool ok = ((int)r.size() == expectedLen) && incr && isSubseq(r, nums);
    cout << "nums=[";
    for (size_t k = 0; k < nums.size(); ++k) cout << nums[k] << (k+1 < nums.size() ? "," : "");
    cout << "] -> [";
    for (size_t k = 0; k < r.size(); ++k) cout << r[k] << (k+1 < r.size() ? "," : "");
    cout << "] (len=" << r.size() << ", expected " << expectedLen << ")"
         << (ok ? " OK" : " FAIL") << '\n';
};

run_pLIS({},                          0);
run_pLIS({5},                         1);
run_pLIS({1, 2, 3, 4, 5},             5);
run_pLIS({5, 4, 3, 2, 1},             1);
run_pLIS({10, 9, 2, 5, 3, 7, 101, 18}, 4);  // e.g., {2,3,7,18}
run_pLIS({0, 1, 0, 3, 2, 3},          4);    // e.g., {0,1,2,3}
run_pLIS({1, 3, 6, 7, 9, 4, 10, 5, 6}, 6);  // {1,3,6,7,9,10}


## 7.3 Number of Longest Increasing Subsequences

**Problem.** Given $\text{nums}$, count the *number* of LISs of the maximum length.

---

### Theory — two parallel DPs

We can't just count strictly-increasing subsequences (there are exponentially many). We want the number of LISs of *maximum* length specifically.

**State definitions.**
- $\text{length}[i]$ = length of LIS ending at $i$ (same as before).
- $\text{count}[i]$ = number of distinct LISs of length $\text{length}[i]$ ending at $i$.

**Invariant.** $\text{count}[i]$ counts each LIS ending at $i$ by its prefix-up-to-$j$ contribution. When we extend any LIS-ending-at-$j$ with $\text{nums}[i]$ as a new last element, we contribute $\text{count}[j]$ new LISs.

**Recurrence (two cases for each predecessor $j$).**

For each $i$, iterate $j$ from $0$ to $i-1$. If $\text{nums}[j] < \text{nums}[i]$:

- **Case 1: $\text{length}[j] + 1 > \text{length}[i]$ — new maximum length found**
  $$\text{length}[i] \leftarrow \text{length}[j] + 1, \quad \text{count}[i] \leftarrow \text{count}[j]$$
  We've discovered a longer subsequence; restart the count from $j$'s contributions.

- **Case 2: $\text{length}[j] + 1 = \text{length}[i]$ — tie at current best**
  $$\text{count}[i] \leftarrow \text{count}[i] + \text{count}[j]$$
  Another way to achieve the same length; add to the count.

- **Case 3: $\text{length}[j] + 1 < \text{length}[i]$ — strictly worse, ignore.**

**Base case.** $\text{length}[i] = 1, \text{count}[i] = 1$ for all $i$ (single-element subsequence).

**Boundary transitions table.**

| Predecessor relation | Update to $(\text{length}[i], \text{count}[i])$ |
|---|---|
| $\text{nums}[j] \ge \text{nums}[i]$ | none (cannot extend) |
| $\text{length}[j] + 1 > \text{length}[i]$ | $(\text{length}[j]+1,\ \text{count}[j])$ |
| $\text{length}[j] + 1 = \text{length}[i]$ | $(\text{length}[i],\ \text{count}[i] + \text{count}[j])$ |
| $\text{length}[j] + 1 < \text{length}[i]$ | unchanged |

**Final answer.** $\sum_{i:\ \text{length}[i] = L} \text{count}[i]$ where $L = \max_i \text{length}[i]$.

**Why it works.**
- Every LIS of length $L$ ends at exactly one index $i^*$. The contribution at $i^*$ is counted in $\text{count}[i^*]$.
- The case-split (Case 1 vs Case 2) preserves the invariant that $\text{count}[i]$ is the count *of subsequences of length $\text{length}[i]$* — not all subsequences ending at $i$. The "restart on improvement" is what enforces this.

**Complexity.** Time $O(n^2)$ (still nested), Space $O(n)$.

**Delta vs LIS.** Adds a parallel count array with the two-case update. Subtle but mechanical once you see the "tie ⟹ add, improve ⟹ reset" pattern.

---

In [ ]:
// Number of LIS — parallel length and count
#include <vector>
#include <algorithm>
#include <iostream>
using namespace std;

int numberOfLIS(const vector<int>& nums) {
    int n = (int)nums.size();
    if (n == 0) return 0;
    vector<int> length(n, 1);                              // length[i] = LIS length ending at i
    vector<int> count(n, 1);                               // count[i]  = # of LISs of that length ending at i
    int maxLen = 1;
    for (int i = 1; i < n; ++i) {
        for (int j = 0; j < i; ++j) {
            if (nums[j] < nums[i]) {                       // strict-increase candidate
                if (length[j] + 1 > length[i]) {
                    length[i] = length[j] + 1;             // new best length: reset count
                    count[i] = count[j];
                } else if (length[j] + 1 == length[i]) {
                    count[i] += count[j];                  // tie: accumulate
                }
                // length[j] + 1 < length[i]: do nothing
            }
        }
        if (length[i] > maxLen) maxLen = length[i];
    }
    int total = 0;
    for (int i = 0; i < n; ++i) if (length[i] == maxLen) total += count[i];
    return total;
}


In [ ]:
// Tests — Number of LIS
auto run_nLIS = [](vector<int> nums, int expected) {
    int a = numberOfLIS(nums);
    cout << "nums=[";
    for (size_t k = 0; k < nums.size(); ++k) cout << nums[k] << (k+1 < nums.size() ? "," : "");
    cout << "] -> " << a << " (Expected: " << expected << ")"
         << (a == expected ? " OK" : " FAIL") << '\n';
};

run_nLIS({},                  0);
run_nLIS({1},                 1);
run_nLIS({1, 3, 5, 4, 7},     2);   // LIS=4: {1,3,5,7} and {1,3,4,7}
run_nLIS({2, 2, 2, 2, 2},     5);   // LIS=1; each of the 5 singletons counts
run_nLIS({1, 2, 4, 3, 5, 4, 7, 2}, 3); // LIS=5: {1,2,4,5,7}, {1,2,3,5,7}, {1,2,3,4,7}
run_nLIS({1, 2, 3, 4, 5},     1);   // unique LIS
run_nLIS({5, 4, 3, 2, 1},     5);   // 5 distinct singletons
run_nLIS({1, 2, 3, 1, 2, 3}, 4);   // LIS=3: {1,2,3} from positions {0,1,2}, {0,1,5}, {0,4,5}, {3,4,5} = 4


## 7.4 Longest Bitonic Subsequence

**Problem.** A *bitonic* sequence is one that is first strictly increasing then strictly decreasing (the peak may be at either end, making pure-increasing and pure-decreasing degenerate bitonic shapes). Given $\text{nums}$, find the length of the longest bitonic subsequence.

---

### Theory — two LIS passes meeting at the peak

**Key observation.** A bitonic subsequence with peak at index $i$ decomposes into:
1. A strictly increasing subsequence ending at $i$ (the "up" half).
2. A strictly decreasing subsequence starting at $i$ (the "down" half).

The peak index $i$ is counted in *both* halves, so we subtract 1 when combining.

**Two parallel state definitions.**
- $\text{LIS}[i]$ = length of longest *increasing* subsequence ending at $i$ (the standard LIS).
- $\text{LDS}[i]$ = length of longest *decreasing* subsequence *starting* at $i$ (equivalently: LIS of the reversed array, repositioned).

**Recurrence.**
- $\text{LIS}[i] = 1 + \max\{0\} \cup \{\text{LIS}[j]\ :\ j < i,\ \text{nums}[j] < \text{nums}[i]\}$ (standard LIS, scan left-to-right).
- $\text{LDS}[i] = 1 + \max\{0\} \cup \{\text{LDS}[k]\ :\ k > i,\ \text{nums}[k] < \text{nums}[i]\}$ (scan right-to-left).

**Answer.**
$$\text{answer} = \max_i\ \big(\text{LIS}[i] + \text{LDS}[i] - 1\big)$$

**Boundary transitions table.**

| Pass | Direction | Recurrence |
|---|---|---|
| LIS | left-to-right | $\text{LIS}[i] = 1 + \max\{\text{LIS}[j] : j < i,\ \text{nums}[j] < \text{nums}[i]\}$ |
| LDS | right-to-left | $\text{LDS}[i] = 1 + \max\{\text{LDS}[k] : k > i,\ \text{nums}[k] < \text{nums}[i]\}$ |
| Combine | — | $\max_i (\text{LIS}[i] + \text{LDS}[i] - 1)$ |

**Why it works.**
- *Decomposition is exhaustive:* any bitonic subsequence has exactly one peak (the maximum element). Iterating over all candidate peaks $i$ and combining the best "up" and "down" halves visits every possible bitonic subsequence.
- *No double counting:* the $-1$ accounts for the peak being counted in both halves.
- *Degenerate cases:* a pure increasing subsequence has $\text{LDS}[i] = 1$ at the rightmost element, contributing $\text{LIS}[\text{end}] + 1 - 1 = \text{LIS}[\text{end}]$ — i.e., the pure LIS itself.

**Complexity.** Time $O(n^2)$ (two LIS passes), Space $O(n)$ (two arrays).

**Delta vs LIS.** Two parallel DPs, then a combination step. Pure mechanical extension.

---

In [ ]:
// Longest Bitonic Subsequence — LIS from left + LDS from right
#include <vector>
#include <algorithm>
#include <iostream>
using namespace std;

int longestBitonic(const vector<int>& nums) {
    int n = (int)nums.size();
    if (n == 0) return 0;
    vector<int> LIS(n, 1);                                 // LIS ending at i
    vector<int> LDS(n, 1);                                 // LDS starting at i
    // Forward pass: LIS
    for (int i = 1; i < n; ++i) {
        for (int j = 0; j < i; ++j) {
            if (nums[j] < nums[i] && LIS[j] + 1 > LIS[i]) {
                LIS[i] = LIS[j] + 1;                       // standard LIS recurrence
            }
        }
    }
    // Backward pass: LDS — increment k from the right, looking even further right.
    for (int i = n - 2; i >= 0; --i) {
        for (int k = i + 1; k < n; ++k) {
            if (nums[k] < nums[i] && LDS[k] + 1 > LDS[i]) {
                LDS[i] = LDS[k] + 1;                       // LDS = LIS of reversed sequence (semantically)
            }
        }
    }
    // Combine at each peak: LIS[i] + LDS[i] - 1.
    int best = 0;
    for (int i = 0; i < n; ++i) {
        best = max(best, LIS[i] + LDS[i] - 1);
    }
    return best;
}


In [ ]:
// Tests — Longest Bitonic Subsequence
auto run_bit = [](vector<int> nums, int expected) {
    int a = longestBitonic(nums);
    cout << "nums=[";
    for (size_t k = 0; k < nums.size(); ++k) cout << nums[k] << (k+1 < nums.size() ? "," : "");
    cout << "] -> " << a << " (Expected: " << expected << ")"
         << (a == expected ? " OK" : " FAIL") << '\n';
};

run_bit({},                                  0);
run_bit({5},                                 1);
run_bit({1, 2, 3, 4, 5},                     5);   // pure increasing
run_bit({5, 4, 3, 2, 1},                     5);   // pure decreasing
run_bit({1, 11, 2, 10, 4, 5, 2, 1},          6);   // classic: 1,2,10,4,2,1 or 1,11,10,4,2,1
run_bit({12, 11, 40, 5, 3, 1},               5);   // 12,11,5,3,1? No, need increasing first. 12,40,5,3,1 ?
                                                    // 12,40 strict increasing (len 2), then 40,5,3,1 strict decreasing (len 4). Total = 2+4-1 = 5.
run_bit({80, 60, 30, 40, 20, 10},            5);   // 30,40,20,10? 60,30 dec, or 30,40,20,10: peak 40, len 2+3-1=4. Better: 80, 30, 20, 10 ? Or peak at 60: 60,30,20,10 (len 4). Peak at 80: 80,60,30,20,10 (len 5).
run_bit({1, 3, 5, 4, 2},                     5);   // 1,3,5,4,2 itself is bitonic


## 7.5 LIS via Patience Sorting — $O(n \log n)$

**Problem.** Compute LIS length in $O(n \log n)$.

This isn't a "space optimization" of §7.1 — it's a *different algorithm*, with the same $O(n)$ space but a logarithmic-per-element inner step. The construction is one of the most beautiful in classical algorithms.

---

### Derivation from first principles

Our $O(n^2)$ algorithm spends most of its time on the inner loop: for each $i$, scan all $j < i$ to find the best predecessor. Can we organize the predecessors so this query becomes fast?

**Step 1: which predecessor matters?**

Among all subsequences seen so far, group them by length. For each length $\ell$, there may be multiple "best so far" subsequences of length $\ell$. For extending, **the smallest tail value is the most useful** — a smaller tail leaves more room for future extensions.

So we maintain:

> $\text{tails}[\ell]$ = the **minimum** tail value over all increasing subsequences of length $\ell + 1$ seen so far (indexing from 0 for length 1, 1 for length 2, etc.).

**Step 2: the key claim — $\text{tails}$ is strictly increasing.**

**Lemma.** For all valid indices $\ell_1 < \ell_2$, $\text{tails}[\ell_1] < \text{tails}[\ell_2]$.

**Proof.** The witness for $\text{tails}[\ell_2]$ is an increasing subsequence $a_1 < a_2 < \ldots < a_{\ell_2 + 1}$ with $a_{\ell_2 + 1} = \text{tails}[\ell_2]$. Truncate to its first $\ell_1 + 1$ elements: $a_1 < a_2 < \ldots < a_{\ell_1 + 1}$. This is an increasing subsequence of length $\ell_1 + 1$ with tail $a_{\ell_1 + 1} < a_{\ell_2 + 1} = \text{tails}[\ell_2]$. By the minimality of $\text{tails}[\ell_1]$, we get $\text{tails}[\ell_1] \le a_{\ell_1 + 1} < \text{tails}[\ell_2]$.  $\square$

So $\text{tails}$ is a strictly increasing array. **Binary search applies.**

**Step 3: the update step.**

When we encounter a new element $x = \text{nums}[i]$, two possible outcomes:

- **Case A: $x$ is strictly greater than every existing tail.**
  Then we can extend the current longest subsequence (the one with tail $\text{tails}[\text{end}]$) by appending $x$. Push $x$ onto $\text{tails}$. The LIS length grows by 1.

- **Case B: there exists some tail $\ge x$.**
  Find the *smallest* such tail; call its position $\ell^*$. (By the sorted invariant, $\text{tails}[\ell^* - 1] < x \le \text{tails}[\ell^*]$.)
  
  Replace $\text{tails}[\ell^*]$ with $x$.

**Why this replace is correct.**

We claim: after replacing, $\text{tails}[\ell^*]$ remains the minimum tail over all increasing subsequences of length $\ell^* + 1$ seen so far.

Proof sketch:
- There exists an increasing subsequence of length $\ell^*$ ending at some value $\le \text{tails}[\ell^* - 1] < x$. Append $x$ to it to get an increasing subsequence of length $\ell^* + 1$ with tail $x$.
- The old $\text{tails}[\ell^*]$ was the previous minimum, now matched or beaten by $x$.
- We pick the *smaller* of $x$ and the old $\text{tails}[\ell^*]$. Since we're in Case B, $x \le$ old $\text{tails}[\ell^*]$. So $x$ wins. Replace.

**Why we replace the smallest qualifying tail and not a larger one.** Replacing a *larger* tail (at position $> \ell^*$) would break the sorted invariant: we'd have $\text{tails}[\ell^*] > x$ (unchanged) while inserting $x$ at a higher index — that violates monotonicity.

**Step 4: the binary-search primitive.**

For strict-increasing LIS (no ties allowed), we use **`lower_bound`** — find the first element $\ge x$.
- If `lower_bound` returns `end()`: Case A (append).
- Otherwise: Case B (replace at that position).

For non-decreasing LIS (longest non-decreasing subsequence, ties allowed): use **`upper_bound`** instead — find the first element $> x$, so ties are absorbed into the same position.

> **A common mistake:** using `upper_bound` for strict LIS gives the longest non-decreasing subsequence (which can be longer than strict LIS). The distinction is one character of code but matters for correctness.

**Complexity.** Time $O(n \log n)$. Space $O(n)$ for `tails`.

**The patience-card-game intuition (brief).**

Deal the cards $\text{nums}[0], \text{nums}[1], \ldots$ one by one into piles, by this rule: place the new card on the *leftmost* pile whose top is $\ge x$, or start a new pile if none qualifies. The number of piles at the end equals the LIS length. The tops of the piles in left-to-right order are exactly the $\text{tails}$ array. (This is the "patience" solitaire pattern, hence the name.)

**Caveat on reconstruction.**

The $\text{tails}$ array does **not** contain the actual LIS — it's a length-tracking artifact whose contents may be a mix of values from different subsequences. To reconstruct the LIS in $O(n \log n)$, you need additional bookkeeping: for each element $x$ placed at position $\ell$ in $\text{tails}$, record a back-pointer to the previous tail (the one at position $\ell - 1$ at the time of insertion). Then walk back from the final position. We omit reconstruction here for clarity.

**Delta vs $O(n^2)$ LIS.** The $O(n)$ space stays. The inner $\max$ over predecessors becomes a $\log n$ binary search. The invariant on $\text{tails}$ does all the work.

---

In [ ]:
// LIS via Patience Sorting — O(n log n)
#include <vector>
#include <algorithm>
#include <iostream>
using namespace std;

// Strict-increasing LIS — use lower_bound (find first ≥ x).
int lis_nlogn(const vector<int>& nums) {
    vector<int> tails;                                     // tails[ℓ] = min tail over length-(ℓ+1) increasing subsequences
    for (int x : nums) {
        // Find smallest position whose tail is ≥ x. If none, append; else replace.
        auto it = lower_bound(tails.begin(), tails.end(), x);
        if (it == tails.end()) {
            tails.push_back(x);                            // Case A: extend
        } else {
            *it = x;                                       // Case B: replace
        }
    }
    return (int)tails.size();
}

// Non-decreasing LIS (longest non-decreasing subsequence) — same algorithm but with upper_bound.
// Provided for contrast; not needed for the strict LIS problem.
int lndss_nlogn(const vector<int>& nums) {
    vector<int> tails;
    for (int x : nums) {
        auto it = upper_bound(tails.begin(), tails.end(), x);   // first > x; ties absorbed
        if (it == tails.end()) tails.push_back(x);
        else *it = x;
    }
    return (int)tails.size();
}


In [ ]:
// Tests — LIS via Patience Sorting
auto run_lisN = [](vector<int> nums, int expected) {
    int a = lis_nlogn(nums);
    // Cross-check against the O(n^2) version.
    int b = lis_tab(nums);
    bool ok = (a == expected && b == expected);
    cout << "nums=[";
    for (size_t k = 0; k < nums.size(); ++k) cout << nums[k] << (k+1 < nums.size() ? "," : "");
    cout << "] -> n_log_n=" << a << " n^2=" << b
         << " (Expected: " << expected << ")"
         << (ok ? " OK" : " FAIL") << '\n';
};

run_lisN({},                          0);
run_lisN({5},                         1);
run_lisN({1, 2, 3, 4, 5},             5);
run_lisN({5, 4, 3, 2, 1},             1);
run_lisN({2, 2, 2, 2},                1);   // strict: ties don't extend
run_lisN({10, 9, 2, 5, 3, 7, 101, 18}, 4);
run_lisN({0, 1, 0, 3, 2, 3},          4);
run_lisN({1, 3, 6, 7, 9, 4, 10, 5, 6}, 6);
run_lisN({3, 1, 8, 2, 5},             3);   // {1,2,5} or {1,8} etc; max len 3

// Also exercise the non-decreasing variant for comparison
cout << "\n-- Non-decreasing (ties allowed) --\n";
auto run_lndss = [](vector<int> nums, int expected) {
    int a = lndss_nlogn(nums);
    cout << "nums=[";
    for (size_t k = 0; k < nums.size(); ++k) cout << nums[k] << (k+1 < nums.size() ? "," : "");
    cout << "] -> lndss=" << a << " (Expected: " << expected << ")"
         << (a == expected ? " OK" : " FAIL") << '\n';
};
run_lndss({2, 2, 2, 2},                4);   // non-decreasing: ties extend
run_lndss({1, 3, 2, 3, 3, 4},          5);   // {1,2,3,3,4}


## 7.6 Longest Divisible Subset

**Problem.** Given a set of distinct positive integers, find the largest subset $S$ such that for every pair $(a, b) \in S$, either $a \mid b$ or $b \mid a$. Return the actual subset.

---

### Theory — sort, then LIS-shaped DP with divisibility

**Key reduction.** If we sort the array ascending, divisibility becomes a one-way relation: for $a < b$ in the sorted array, "$a \mid b$ or $b \mid a$" simplifies to "$a \mid b$" (since $b > a > 0$ rules out $b \mid a$).

Furthermore, divisibility is **transitive on sorted positive integers**: if $a \mid b$ and $b \mid c$ (with $a < b < c$), then $a \mid c$. So any chain $a_1 \mid a_2 \mid \ldots \mid a_k$ (with strict increase) is also pairwise divisible — every $a_i$ divides every $a_j$ with $i < j$.

This means: **a divisible subset, after sorting, is a chain of divisibilities** — exactly the LIS pattern, with `<` replaced by `divides`.

**State definition.** After sorting $\text{nums}$ ascending:
$\text{dp}[i]$ = size of the largest divisible subset *ending at* $\text{nums}[i]$ (using $\text{nums}[i]$ as the largest element).

**Recurrence.**
$$\text{dp}[i] = 1 + \max\!\Big(\ \{0\}\ \cup\ \{\text{dp}[j]\ :\ j < i,\ \text{nums}[i] \bmod \text{nums}[j] = 0\}\ \Big)$$

**Base case.** $\text{dp}[i] = 1$ (singleton).

**Boundary transitions table.**

| Case | Recurrence |
|---|---|
| $i = 0$ | $\text{dp}[0] = 1$ |
| $\nexists j$ with $\text{nums}[j] \mid \text{nums}[i]$ | $\text{dp}[i] = 1$ |
| $\exists j$ with $\text{nums}[j] \mid \text{nums}[i]$ | $\text{dp}[i] = 1 + \max \text{dp}[j]$ over valid $j$ |

**Reconstruction.** Same parent-pointer technique as Print LIS — record which $j$ achieved the winning value, then walk backwards from $\arg\max_i \text{dp}[i]$.

**Why it works.**
- *Optimal substructure:* dropping the largest element of a max-size divisible subset gives a max-size divisible subset over the remaining elements ending at the second-largest. (Standard exchange argument.)
- *Sorting is essential:* without sorting, the recurrence "look for valid predecessors" requires considering all *unordered* pairs, breaking the linear scan. After sorting, the "ending here" anchor works cleanly because we only check earlier (smaller) elements.

**Complexity.** Time $O(n \log n)$ for sorting + $O(n^2)$ for the DP. Space $O(n)$.

**Delta vs LIS.** Same recurrence shape, with the binary relation changed from `<` to `divides`. The sort is the bridge that makes the linear-scan structure apply.

---

In [ ]:
// Longest Divisible Subset — sort + LIS-shaped DP with divisibility + reconstruction
#include <vector>
#include <algorithm>
#include <iostream>
using namespace std;

vector<int> largestDivisibleSubset(vector<int> nums) {
    int n = (int)nums.size();
    if (n == 0) return {};
    sort(nums.begin(), nums.end());                        // crucial: divisibility becomes one-directional after sort
    vector<int> dp(n, 1);                                  // dp[i] = max divisible chain ending at nums[i]
    vector<int> parent(n, -1);                             // for reconstruction
    int bestIdx = 0;
    for (int i = 1; i < n; ++i) {
        for (int j = 0; j < i; ++j) {
            // nums[j] < nums[i] (by sort); divisibility check reduces to nums[i] % nums[j] == 0
            if (nums[i] % nums[j] == 0 && dp[j] + 1 > dp[i]) {
                dp[i] = dp[j] + 1;
                parent[i] = j;
            }
        }
        if (dp[i] > dp[bestIdx]) bestIdx = i;
    }
    // Walk back via parents
    vector<int> result;
    for (int i = bestIdx; i != -1; i = parent[i]) result.push_back(nums[i]);
    reverse(result.begin(), result.end());
    return result;
}


In [ ]:
// Tests — Longest Divisible Subset
auto run_lds = [](vector<int> nums, int expectedLen) {
    vector<int> r = largestDivisibleSubset(nums);
    // Validate: every pair in r is mutually divisible, and length matches
    bool pairOk = true;
    for (size_t a = 0; a < r.size() && pairOk; ++a)
        for (size_t b = a + 1; b < r.size() && pairOk; ++b)
            if (r[a] % r[b] != 0 && r[b] % r[a] != 0) pairOk = false;
    bool ok = ((int)r.size() == expectedLen) && pairOk;
    cout << "nums=[";
    for (size_t k = 0; k < nums.size(); ++k) cout << nums[k] << (k+1 < nums.size() ? "," : "");
    cout << "] -> [";
    for (size_t k = 0; k < r.size(); ++k) cout << r[k] << (k+1 < r.size() ? "," : "");
    cout << "] (len=" << r.size() << ", expected " << expectedLen << ")"
         << (ok ? " OK" : " FAIL") << '\n';
};

run_lds({},                             0);
run_lds({1},                            1);
run_lds({1, 2, 3},                      2);   // {1,2} or {1,3}
run_lds({1, 2, 4, 8},                   4);   // perfect chain
run_lds({1, 2, 4, 8, 16, 32},           6);   // longer perfect chain
run_lds({3, 4, 16, 8},                  3);   // sorted: {3,4,8,16}; chain {4,8,16} len 3
run_lds({2, 3, 4, 9, 8},                3);   // {2,4,8}
run_lds({5, 9, 18, 54, 108, 540, 90, 180, 360, 720}, 6);  // {9,18,90,180,360,720}


## 7.7 Russian Doll Envelopes

**Problem.** Given $n$ envelopes as pairs $(w_i, h_i)$. Envelope $(w_1, h_1)$ fits inside $(w_2, h_2)$ iff $w_1 < w_2$ **and** $h_1 < h_2$ (strict on both). Find the maximum number of envelopes that can be nested.

---

### Theory — 2D LIS via sort + tie-break trick

**The naive attempt.** Sort by $w$ ascending; then we need to find LIS on $h$. But this fails when two envelopes share the same $w$ — they can't fit inside each other, but a naive sort might process them in $h$-ascending order, fooling the LIS into picking up two of them.

**The fix — sort with a careful tie-break.**

> **Sort envelopes by $w$ ascending, with ties on $w$ broken by $h$ *descending*.**

**Why descending on ties.** Suppose two envelopes have $w_1 = w_2 = W$ with $h_1 < h_2$. Tie-breaking by $h$ ascending places $h_1$ before $h_2$. Then LIS on $h$ might pick both, treating it as a valid nesting — wrong, because $w_1 = w_2$ violates strict inequality.

Tie-breaking by $h$ descending places $h_2$ before $h_1$. Now LIS on $h$, scanning left-to-right, sees $h_2$ first, then $h_1 < h_2$. Since LIS requires strict increase, it *cannot* pick both. Exactly one is included — correct.

**Algorithm.**
1. Sort envelopes by $(w \uparrow,\ h \downarrow)$.
2. Extract the $h$ values into an array.
3. Compute the strict LIS on the $h$ array (use the $O(n \log n)$ patience sorting).
4. Return the LIS length.

**Why $O(n \log n)$ here.** With $n$ up to $10^5$ (LeetCode), the $O(n^2)$ DP is too slow. The patience sorting algorithm from §7.5 is the key tool — that's why we covered it first.

**Boundary transitions table.**

| Step | Action |
|---|---|
| Sort | $(w \uparrow,\ h \downarrow)$ |
| Extract | $h$-sequence |
| LIS | strict (use `lower_bound`) |

**Why it works.**
- *Sorting reduces 2D to 1D:* after sort, the $w$-constraint is automatically satisfied for any left-to-right LIS pick (positions are sorted, and the strict-$h$-increase combined with the descending tie-break enforces strict-$w$ as a side effect).
- *Strictness on $w$:* the descending tie-break is the entire device that prevents LIS-on-$h$ from selecting two same-$w$ envelopes.

**Complexity.** Time $O(n \log n)$ (sort + LIS). Space $O(n)$.

**Delta vs LIS.** A pre-processing sort plus a single LIS call. The intellectual content is the tie-break — without it, the answer is silently wrong.

---

In [ ]:
// Russian Doll Envelopes — sort + O(n log n) LIS on h
#include <vector>
#include <algorithm>
#include <iostream>
using namespace std;

int russianDoll(vector<vector<int>>& envelopes) {
    int n = (int)envelopes.size();
    if (n == 0) return 0;
    // Sort: w ascending, h DESCENDING on ties.
    // The descending tie-break prevents same-w envelopes from being chained by LIS-on-h.
    sort(envelopes.begin(), envelopes.end(),
         [](const vector<int>& a, const vector<int>& b) {
             if (a[0] != b[0]) return a[0] < b[0];         // w ascending
             return a[1] > b[1];                            // h descending on ties
         });
    // Extract h values and compute strict LIS.
    vector<int> h(n);
    for (int i = 0; i < n; ++i) h[i] = envelopes[i][1];
    return lis_nlogn(h);                                    // reuse patience-sorting LIS
}


In [ ]:
// Tests — Russian Doll Envelopes
auto run_rd = [](vector<vector<int>> envs, int expected) {
    vector<vector<int>> copy = envs;                       // sort modifies in place
    int a = russianDoll(copy);
    cout << "envs={";
    for (size_t k = 0; k < envs.size(); ++k) {
        cout << "(" << envs[k][0] << "," << envs[k][1] << ")";
        if (k+1 < envs.size()) cout << ",";
    }
    cout << "} -> " << a << " (Expected: " << expected << ")"
         << (a == expected ? " OK" : " FAIL") << '\n';
};

run_rd({},                                            0);
run_rd({{1, 1}},                                       1);
run_rd({{5, 4}, {6, 4}, {6, 7}, {2, 3}},               3);   // LeetCode: {2,3} → {5,4} → {6,7}
run_rd({{1, 1}, {1, 1}, {1, 1}},                       1);   // all same → cannot nest
run_rd({{4, 5}, {4, 6}, {6, 7}, {2, 3}, {1, 1}},       4);   // {1,1}→{2,3}→{4,5}→{6,7}? But also {4,6}.
                                                              // After tie-break sort: (1,1), (2,3), (4,6), (4,5), (6,7)
                                                              // h sequence: 1, 3, 6, 5, 7 → LIS on h: {1,3,6,7} or {1,3,5,7} = 4
run_rd({{30, 50}, {12, 2}, {3, 4}, {12, 15}},          3);   // {3,4}→{12,15}→{30,50} = 3
run_rd({{1, 3}, {3, 5}, {6, 7}, {6, 8}, {8, 4}, {9, 5}}, 3); // {1,3}→{3,5}→{6,7} or {6,8}


# Unified Mental Model — DP on LIS family

All seven problems share the same anchoring trick — **"ending exactly at index $i$"** — applied to different orderings or supplemented with extra structure.

---

In [ ]:
// ============================================================
// THE UNIFIED LIS SKELETON
// ============================================================
//
// State:  dp[i] = answer for subsequence ending exactly at index i
// Recurrence:
//     dp[i] = base + best_combine( dp[j] : j < i, relation(nums[j], nums[i]) )
//
// Where:
//   base ∈ {1, 0, ...}        — singleton's contribution
//   relation                   — < (LIS), divides (Divisible Subset), or composed (Russian Doll)
//   best_combine               — max (length), sum (counting), max+1 (length+1)
//
// Answer:  max over i of dp[i]  (or aggregate, e.g. sum)
//
// ============================================================
// INSTANTIATIONS
// ============================================================
//
// LIS (§7.1)                     relation = <       combine = max(dp[j])+1            answer = max dp
// Print LIS (§7.2)               LIS DP + parent[]  reconstruct by walking parents     answer = sequence
// Number of LIS (§7.3)           LIS DP + count[]   two cases (improve vs tie)          answer = sum count
// Longest Bitonic (§7.4)         LIS + LDS          combine: LIS[i] + LDS[i] - 1         answer = max combined
// Longest Divisible (§7.6)       sort first         relation = nums[i] % nums[j] == 0    answer = max dp + reconstruct
// Russian Doll (§7.7)            2D → sort + LIS    sort (w↑, h↓), LIS on h              answer = LIS length
//
// LIS patience sorting (§7.5)    DIFFERENT ALGORITHM — not a DP recurrence on dp[i].
//                                Maintains tails[ℓ] = min tail over length-(ℓ+1) increasing subseqs.
//                                tails is sorted, so binary search applies.
//
// ============================================================
// PATIENCE SORTING IN ONE BOX
// ============================================================
//
//   for x in nums:
//       it = lower_bound(tails, x)          // first ≥ x  (lower_bound for STRICT; upper_bound for ≥)
//       if it == end:  tails.push_back(x)   // x extends the longest subsequence
//       else:          *it = x              // x improves a tail (lowers it) at position it
//   answer = tails.size()
//
// Invariants:
//   • tails is strictly increasing throughout
//   • tails[ℓ] = min tail over all length-(ℓ+1) increasing subseqs of nums[0..i] so far
//   • tails.size() = LIS length so far
//
// ============================================================
// SPACE-OPT TABLE
// ============================================================
//   LIS                : O(n) (full predecessor history needed in scan)
//   LIS n log n        : O(n) for tails array
//   Print LIS          : O(n) for dp + O(n) for parent
//   Number of LIS      : O(n) for length + O(n) for count
//   Bitonic            : O(n) + O(n) for LIS and LDS
//   Divisible Subset   : O(n) + O(n) for reconstruction
//   Russian Doll       : O(n) for sorted h + O(n) tails
// ============================================================


# Decision Tree — recognizing LIS-family problems

```
                ┌──────────────────────────────────────────────────────────┐
                │ Question asks for longest/count of a subsequence whose   │
                │ adjacent elements satisfy some relation (<, divides, …)? │
                └────────────────────┬─────────────────────────────────────┘
                                     │ yes
                                     ▼
                ┌──────────────────────────────────────────────────────────┐
                │ Set up: dp[i] = answer for subsequence ending at i.       │
                │ Recurrence scans j < i with relation(nums[j], nums[i]).   │
                └────────────────────┬─────────────────────────────────────┘
                                     ▼
                ┌──────────────────────────────────────────────────────────┐
                │ Question flavor (trichotomy):                            │
                │  • Max length              → max(dp[j])+1                │
                │  • Count of max-length     → 2 DPs: length[] and count[] │
                │  • Reconstruct one         → parent[] + walk back        │
                │  • Bidirectional (bitonic) → LIS + LDS, combine at peak  │
                └────────────────────┬─────────────────────────────────────┘
                                     ▼
                ┌──────────────────────────────────────────────────────────┐
                │ Is the relation NOT just <?                              │
                │  • Divisibility       → sort first, relation = a | b     │
                │  • 2D nesting         → sort by 1st dim with tie-break,  │
                │                          LIS on 2nd dim                  │
                └────────────────────┬─────────────────────────────────────┘
                                     ▼
                ┌──────────────────────────────────────────────────────────┐
                │ Is n large enough that O(n²) is too slow?                │
                │  • Need just LENGTH                → patience sorting    │
                │  • Need reconstruction / count     → O(n²) plus extras   │
                │  • Need with custom relation       → typically O(n²)     │
                └──────────────────────────────────────────────────────────┘
```

**Three traps to avoid:**

1. **Answer is at $\arg\max$, not at $\text{dp}[n-1]$.** LIS can end at any index. Forgetting the final scan is a silent bug — code "works" but returns the LIS ending at the last position only.

2. **`lower_bound` vs `upper_bound` in patience sorting.** Strict LIS uses `lower_bound`; non-decreasing uses `upper_bound`. A one-character swap that gives a different answer when ties are present.

3. **Russian Doll's tie-break direction.** Sorting by $(w \uparrow, h \uparrow)$ silently double-counts same-$w$ envelopes. The descending tie-break on $h$ is what prevents the LIS from chaining them together. Easy to get wrong and hard to catch from random tests — most inputs don't have $w$-ties.

---

# Complexity Summary

| Problem | Time | Space | Key Insight |
|---|---|---|---|
| **LIS (standard)** | $O(n^2)$ | $O(n)$ | "ending at $i$" anchor; scan $j < i$ |
| **Print LIS** | $O(n^2)$ | $O(n)$ | Parent pointers; walk from $\arg\max$ |
| **Number of LIS** | $O(n^2)$ | $O(n)$ | Two parallel DPs (length, count); two-case update |
| **Longest Bitonic** | $O(n^2)$ | $O(n)$ | LIS + LDS; combine at peak with $-1$ |
| **LIS patience sorting** | $O(n \log n)$ | $O(n)$ | `tails[]` strictly sorted; binary search |
| **Longest Divisible Subset** | $O(n^2 + n \log n)$ | $O(n)$ | Sort first; relation = `a divides b` |
| **Russian Doll Envelopes** | $O(n \log n)$ | $O(n)$ | Sort with descending tie-break on $h$; LIS on $h$ |

---

# Closing Notes

**What you should now be able to do without reaching for any reference:**

1. State the LIS recurrence in one sentence and write it as code in twenty seconds.
2. Recognize an LIS-shape problem from the words "longest" + "subsequence" + "relation between adjacent picked elements" (whether that's $<$, divisibility, fits-inside, etc.).
3. Choose between $O(n^2)$ and $O(n \log n)$ based on whether you need reconstruction, counting, or just length.
4. Prove the patience sorting invariant on demand: `tails[ℓ]` is the minimum tail over all length-$(\ell+1)$ increasing subsequences, and `tails[]` is strictly sorted.
5. Apply the "sort + tie-break + LIS" pattern to 2D nesting problems.

**A unifying perspective: LIS as a longest path on a DAG.**

Build a DAG with $n$ nodes (one per array index) and edges $(j, i)$ for $j < i$ with $\text{nums}[j] < \text{nums}[i]$ (or whatever relation you have). The LIS length equals the longest path in this DAG, plus 1.

- $O(n^2)$ DP: standard longest-path-on-DAG by topological order.
- $O(n \log n)$ patience sorting: a clever indirect computation that *never builds the DAG* explicitly — it exploits the structure of "increasing" to derive the answer from sorted invariants.

The patience sorting algorithm is special precisely because for the "less than" relation specifically, the DAG has hidden structure (it's a comparability graph of a sequence) that allows the $\log n$ trick. For arbitrary relations (e.g., divisibility), no such trick exists, and we fall back to $O(n^2)$. This is why §7.6 (Divisible Subset) is $O(n^2)$ even though it looks structurally like LIS — the relation isn't a total order.

**The discipline.** Eight of ten string problems reduced to LCS in Subtopic 5; six of seven problems here reduce to LIS or its patience-sorting cousin. The pattern is clear: subsequence/subset problems on linear inputs have a small number of "kernel DPs," and most variants are kernel + twist. Memorize the kernels; let the twists be twists.

**Looking ahead — Subtopic 8 (MCM / Partition DP).** A different genre entirely: the state shifts from a single index to a **range $[l, r]$**, and the recurrence case-splits on **where to make the partition** (the split point $k$). The state DAG has a fundamentally different shape — intervals fill *by length* outward, not by position. This is interval DP, and it has its own iteration discipline that we'll work through.